# agent-memory-staleness-audit — quickstart

Finds agent memories that have quietly gone stale: facts that were true when stored and are
silently wrong now, **with nothing in the memory store to contradict them**.

This notebook runs end-to-end with **no API key, no credentials, and no network calls** after
the install cell. Everything below scores a synthetic corpus whose correct answers are known by
construction, so you can check the tool against a designed answer rather than a vibe.

> **What this notebook does not show:** how well the scorer does against a real memory store.
> That evaluation (STALE, arXiv 2605.06527) has not been run yet. Nothing here is a claim about
> Mem0, Zep, or any other product.


## 1. Install


In [ ]:
!pip install -q git+https://github.com/a-bhimava/agent-memory-staleness-audit.git

import memory_staleness
print('version', memory_staleness.__version__)


## 2. The problem, in three memories

Consider what a contradiction detector sees versus what is actually true.


In [ ]:
import datetime as dt
from memory_staleness.types import MemoryEntry, Provenance, FactType

NOW = dt.datetime(2026, 8, 21, 12, 0, tzinfo=dt.UTC)

def memory(mid, subject, predicate, obj, fact_type, days_ago, retrievals):
    return MemoryEntry(
        memory_id=mid, subject=subject, predicate=predicate, object=obj,
        fact_type=fact_type, retrieval_count=retrievals,
        provenance=Provenance(source='conversation:1', written_at=NOW - dt.timedelta(days=days_ago)),
    )

entries = (
    # Stale on age alone, and still being retrieved. The dangerous one.
    memory('m1', 'alice', 'works_at', 'Acme', FactType.EMPLOYER, days_ago=1200, retrievals=14),
    # Ancient, but a date of birth does not go stale. Must NOT fire.
    memory('m2', 'alice', 'born_on', '1991-03-14', FactType.DATE_OF_BIRTH, days_ago=7000, retrievals=9),
    # Long dead and nothing reads it. Forget, don't re-verify.
    memory('m3', 'alice', 'working_on', 'Project Zeus', FactType.PROJECT_STATUS, days_ago=600, retrievals=0),
)
for e in entries:
    print(f'{e.memory_id}  {e.text:38}  {e.fact_type.value}')


### Score them


In [ ]:
from memory_staleness.scoring import score_store

for s in score_store(entries, as_of=NOW):
    print(f'{s.memory_id}  {s.verdict.value.upper():14} score={s.score:.3f}')
    for reason in s.reasons:
        print(f'        - {reason}')
    print()


Note `m2`. An age-only scorer flags it — it is nineteen years old and heavily retrieved.
It scores **0.000** here because the volatility table says a date of birth does not decay.
That specificity is the difference between an auditor people use and one they learn to ignore.


## 3. The case that motivates the project

Two entries, same subject and predicate, and the newer one **agrees** with the older one.
There is no contradiction anywhere — so contradiction-based invalidation has nothing to fire on.


In [ ]:
silent = (
    memory('old', 'bob', 'works_at', 'Globex', FactType.EMPLOYER, days_ago=900, retrievals=7),
    memory('new', 'bob', 'works_at', 'Globex', FactType.EMPLOYER, days_ago=30, retrievals=2),
)

for s in score_store(silent, as_of=NOW):
    flag = s.supersession
    print(f'{s.memory_id}  {s.verdict.value.upper():14} score={s.score:.3f}')
    if flag:
        print(f'        superseded by {flag.superseded_by!r}; contradicts={flag.contradicts}')


`contradicts=False` is the whole point. Zep/Graphiti's bi-temporal invalidation and Mem0's
supersession both need a disagreement to trigger. Here there isn't one.


## 4. Known-answer validation

The synthetic generator plants entries **with their expected verdict attached**, so accuracy
can be measured without anyone labelling anything. Half the corpus is deliberately clean —
an auditor that flags everything has perfect recall and zero worth.


In [ ]:
from collections import Counter
from memory_staleness.synth import generate_store

store = generate_store(n_per_kind=8)
scored = {s.memory_id: s for s in score_store(store.entries, as_of=store.as_of)}

correct = sum(1 for c in store.cases if scored[c.entry.memory_id].verdict is c.expected)
print(f'{correct}/{len(store.cases)} planted cases got their expected verdict')
print()
for kind, n in sorted(Counter(c.kind.value for c in store.cases).items()):
    hits = sum(1 for c in store.cases
               if c.kind.value == kind and scored[c.entry.memory_id].verdict is c.expected)
    print(f'  {kind:34} {hits}/{n}')


> 100% here is the **floor, not an achievement**. The answers are known by construction, so
> anything less would mean the scorer is broken rather than merely weak. The real test is the
> benchmark run, which has not happened yet.


## 5. The volatility table

The load-bearing part of the design, and deliberately hand-authored rather than model-inferred —
so a reviewer can point at one number and disagree with it.


In [ ]:
from memory_staleness.volatility import load_table

table = load_table()
rows = sorted(table.entries, key=lambda e: (e.half_life_days is None, e.half_life_days or 0))
for e in rows:
    half_life = 'never decays' if e.half_life_days is None else f'{e.half_life_days:>6.0f}d'
    print(f'{e.fact_type.value:24} {half_life}')
print()
print('table digest:', table.source_sha256)


Every score in an audit depends on these numbers, which is why the digest above is recorded
in the run manifest. A result that does not name its table is not reproducible.


## 6. The full pipeline: run → export → verify

`export` buffers every byte and checks it before writing. `verify` **re-derives** the bundle
rather than re-diffing it — re-exporting and diffing proves the exporter is deterministic;
re-deriving proves it is correct.


In [ ]:
!staleness-audit run --synthetic --n-per-kind 8
!staleness-audit export --out /content/bundle
!staleness-audit verify /content/bundle --strict


## 7. Auditing your own store

Export your memories to JSONL with one object per line, then point the CLI at it. No store
client is required for this path — the adapters read exports, not live databases.

```json
{"memory_id": "1", "subject": "alice", "predicate": "works_at", "object": "Acme",
 "fact_type": "employer", "retrieval_count": 12,
 "provenance": {"source": "crm-sync", "written_at": "2024-02-01T00:00:00Z"}}
```

```bash
staleness-audit run --from-export my_memories.jsonl
```

Entries with no `written_at` return `CANNOT_ASSESS` — an admission, never a guess. If most of
your store comes back that way, that is itself the finding: provenance is not being captured
upstream.
